In [ ]:
print("ok kr")

In [ ]:
%pwd

In [ ]:
import os
os.chdir("../")

In [ ]:
%pwd

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
def load_pdf_files(pdf_directory):
    loader = DirectoryLoader(pdf_directory, glob="*.pdf", loader_cls=PyPDFLoader)  # ✅ Use the parameter
    documents = loader.load()
    return documents

extracted_documents = load_pdf_files("D:/UNIVERSITY/PYTHON/langchain/MEDICAL-CHATBOT/data")

In [ ]:
# extracted_documents
len(extracted_documents)


In [ ]:
from typing import List
from langchain_core.documents import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """Given a list of documents, filter them to only include the page content and metadata."""
    minimal_docs = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src} if src else {}
            )
        )
    
    return minimal_docs

In [ ]:
minimal_docs = filter_to_minimal_docs(extracted_documents)

In [ ]:
minimal_docs

In [ ]:
# split the documents into smaller chunks
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    texts_chunk = text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [ ]:
chunks = text_split(minimal_docs)
print(f"Number of chunks: {len(chunks)}")

In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings


def download_embeddings():
    """Download the HuggingFace embeddings model and return the embeddings object."""
    
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(model_name=model_name)
    return embeddings

embeddings = download_embeddings()

In [ ]:
embeddings

In [ ]:
vector = embeddings.embed_query("What is the capital of France?")
print(vector)

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

In [ ]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
key = PINECONE_API_KEY
print(key)
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

In [ ]:
from pinecone import Pinecone
Pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=Pinecone_api_key)
pc

In [ ]:
from pinecone import ServerlessSpec

index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,  # Dimension of the embeddings
        metric="cosine",  # Similarity metric
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

In [ ]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name=index_name,
    pinecone_api_key= os.getenv("PINECONE_API_KEY")
)

In [ ]:
# load existing index

from langchain_pinecone import PineconeVectorStore

# embed each chunk and upsert into your Pinecone index
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings,
)

### Add more data to existing index

In [ ]:
zia_data = Document(page_content="Shareeeef bcha hooon mein, yaoh nhi", metadata={"source": "test"})
docsearch.add_documents(documents = [zia_data])

In [ ]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})
docs = retriever.invoke("What is shareef bacha?")
docs
context = "\n".join([doc.page_content for doc in docs])

In [ ]:
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.1,
    max_tokens=1024
)

In [ ]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
from langchain_core.prompts import ChatPromptTemplate


system_prompt = """
You are an assistant for giving answers to questions.
Use the following retrieved documents to answer the question.
If you don't know the answer, say you don't know, use maximum 3 sentences to answer the question and make answers clear and concise and clean.

{context}  # <-- leave as a placeholder
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}")  # user question
    ]
)

In [ ]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [ ]:
response = rag_chain.invoke({"input": "What is acne?"})
print(response["answer"])